In [1]:

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [2]:
# Download dataset
!kaggle datasets download -d uciml/sms-spam-collection-dataset

Dataset URL: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset
License(s): unknown
100% 211k/211k [00:00<00:00, 88.3MB/s]



In [3]:
# Unzip
!unzip sms-spam-collection-dataset.zip

Archive:  sms-spam-collection-dataset.zip
  inflating: spam.csv                


In [4]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [5]:
df = pd.read_csv('spam.csv', encoding='latin-1')

df = df[['v1', 'v2']]
df.columns = ['label', 'message']

# Convert label to numbers
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
stop_words = set(stopwords.words('english'))
stop_words.discard('not')

lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"n't", " not", text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

df['clean_text'] = df['message'].apply(clean_text)

In [7]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9
)

X = tfidf.fit_transform(df['clean_text'])
y = df['label']

In [8]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
lr = LogisticRegression()

params = {
    'C': [0.1, 0.5, 1, 2, 4],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [1000, 2000],
    'class_weight': [None, 'balanced']
}

random = RandomizedSearchCV(
    lr,
    param_distributions=params,
    n_iter=10,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)

random.fit(x_train, y_train)

best_model = random.best_estimator_

In [11]:
y_pred = best_model.predict(x_test)
y_train_pred = best_model.predict(x_train)

print("Train_Accuracy:", accuracy_score(y_train, y_train_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Train_Accuracy: 0.9977563383441777
Accuracy: 0.9820627802690582
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       965
           1       0.95      0.91      0.93       150

    accuracy                           0.98      1115
   macro avg       0.97      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [15]:
user_input = input("Enter email: ")

clean = [clean_text(user_input)]
vec = tfidf.transform(clean)

pred = best_model.predict(vec)

if pred[0] == 1:
    print("Spam ❌")
else:
    print("Not Spam ✅")

Enter email: hello, how are you
Not Spam ✅


In [14]:
user_input = input("Enter email: ")

clean = [clean_text(user_input)]
vec = tfidf.transform(clean)

pred = best_model.predict(vec)

if pred[0] == 1:
    print("Spam ❌")
else:
    print("Not Spam ✅")

Enter email: "Congratulations! You have won a free lottery. Claim now!"
Spam ❌
